In [4]:
import numpy as np
from pathlib import Path
import sys

PROJECT_DIR = str(Path.cwd().parent) + "/"
sys.path.append(PROJECT_DIR)

from mpp_project.oracle_dp import compute_full_Q_table, solve_dp_with_full_empirical_distribution, GRID_SIZE, GAP_OFFSET

print("🧪 DÉMARRAGE DU LABORATOIRE DP : PHASE DE POULES (NOTEBOOK 10)")

# ==========================================
# 1. PARAMÈTRES DU MATCH "PIPEAU"
# ==========================================
true_proba = np.array([0.60, 0.25, 0.15], dtype=np.float32) 
crowd = np.array([0.70, 0.20, 0.10], dtype=np.float32)
gains = np.array([20, 50, 90], dtype=np.int32) 

p_empirique_1D_single = np.zeros((1, 3, 250), dtype=np.float32)
p_empirique_1D_single[0, :, 0] = 1.0 

# ==========================================
# 2. CRÉATION DU "FAUX HORIZON" 
# ==========================================
V_horizon_6D = np.zeros((GRID_SIZE, GRID_SIZE, 2, 2, 2, 2), dtype=np.float32)

for g1 in range(GRID_SIZE):
    for g2 in range(GRID_SIZE):
        # CORRECTION 1 : On SOUSTRAIT l'offset pour vérifier la vraie valeur
        if (g1 - GAP_OFFSET) >= 0 and (g2 - GAP_OFFSET) >= 0:
            V_horizon_6D[g1, g2, :, :, :, :] = 1.0

# Tranche 3D pour les fonctions de poules
V_horizon_3D = V_horizon_6D[:, :, :, 1, 1, 1]

# ==========================================
# 3. TEST DE `compute_full_Q_table` (Le Moteur du Jour J)
# ==========================================
print("\n" + "="*50)
print("🔍 PARTIE 1 : TEST DE compute_full_Q_table (1 Match)")
print("="*50)

# On calcule la Q-Table pour ce match
Q_table = compute_full_Q_table(
    t=0, 
    t_prob=true_proba, 
    c_rep=crowd, 
    t_gains=gains, 
    p_empirique_1D=p_empirique_1D_single, 
    alpha=1.0, 
    V_next=V_horizon_3D,
    max_gain=250
)

idx_g2_safe = GRID_SIZE - 1

def test_q_table(nom, gap_reel_bob, theo_sans_booster, theo_avec_booster):
    gap_grid = int(gap_reel_bob)
    # CORRECTION 2 : On ADDITIONNE l'offset pour trouver l'index
    idx_g1 = gap_grid + GAP_OFFSET
    
    wr_base = np.max(Q_table[idx_g1, idx_g2_safe, :, 0])
    wr_keep = np.max(Q_table[idx_g1, idx_g2_safe, :, 1])
    wr_use = np.max(Q_table[idx_g1, idx_g2_safe, :, 2])
    wr_avec_booster_dispo = max(wr_keep, wr_use)
    
    print(f"▶️ TEST : {nom}")
    print(f"   Gap avec Bob : {gap_reel_bob} pts")
    print(f"   [Base] Théorie : {theo_sans_booster*100:05.2f}% | Calcul Q-Table : {wr_base*100:05.2f}%")
    print(f"   [ x2 ] Théorie : {theo_avec_booster*100:05.2f}% | Calcul Q-Table : {wr_avec_booster_dispo*100:05.2f}%")
    
    if abs(wr_base - theo_sans_booster) < 0.001 and abs(wr_avec_booster_dispo - theo_avec_booster) < 0.001:
        print("   ✅ TEST RÉUSSI")
    else:
        print("   ❌ ÉCHEC DU TEST")
    print("-" * 50)


# SCÉNARIO 1 : L'Outsider x2 est le seul levier
wr_theo_out_seul = true_proba[2] * (1.0 - crowd[2])
test_q_table("Sauvetage Outsider x2", gap_reel_bob=-150, theo_sans_booster=0.0, theo_avec_booster=wr_theo_out_seul)

# SCÉNARIO 2 : Le Nul x2 devient le meilleur levier
wr_theo_nul_seul = true_proba[1] * (1.0 - crowd[1])
test_q_table("Le Casse du Siècle (Nul x2 > Out x2)", gap_reel_bob=-95, theo_sans_booster=0.0, theo_avec_booster=wr_theo_nul_seul)


# ==========================================
# 4. TEST DE `solve_dp_with_full_empirical_distribution` (Profondeur)
# ==========================================
print("\n" + "="*50)
print("🌌 PARTIE 2 : TEST DE solve_dp_with_full_... (Multi-Matchs)")
print("="*50)

N_MATCHS = 3

match_probs = np.zeros((N_MATCHS, 3), dtype=np.float32)
crowds = np.zeros((N_MATCHS, 3), dtype=np.float32)
gains_1N2 = np.zeros((N_MATCHS, 3), dtype=np.int32)

for t in range(N_MATCHS):
    match_probs[t] = true_proba
    crowds[t] = crowd
    gains_1N2[t] = gains

p_empirique_1D_multi = np.zeros((N_MATCHS, 3, 250), dtype=np.float32)
p_empirique_1D_multi[:, :, 0] = 1.0 

alphas = np.ones(N_MATCHS, dtype=np.float32)

# Solveur Récursif Théorique Exact (Base uniquement, pour simplifier le test)
def exact_group_stage_wr(t, current_gap):
    if t == N_MATCHS:
        return 1.0 if current_gap >= 0 else 0.0

    max_wr_for_this_match = 0.0
    for my_action in range(3):
        wr_action = 0.0
        for actual_out in range(3):
            p_out = true_proba[actual_out]
            if p_out == 0: continue
            
            my_gain = gains[actual_out] if my_action == actual_out else 0
            
            for bob_action in range(3):
                p_bob = crowd[bob_action]
                if p_bob == 0: continue
                
                bob_gain = gains[actual_out] if bob_action == actual_out else 0
                new_gap = current_gap + my_gain - bob_gain
                
                wr_action += p_out * p_bob * exact_group_stage_wr(t + 1, new_gap)
                
        if wr_action > max_wr_for_this_match:
            max_wr_for_this_match = wr_action
            
    return max_wr_for_this_match

print(f"🧠 Rétro-propagation DP sur {N_MATCHS} matchs...")
V_history_poules = solve_dp_with_full_empirical_distribution(
    # CORRECTION 3 : On passe V_horizon_3D !
    match_probs, crowds, gains_1N2, p_empirique_1D_multi, alphas, V_horizon_3D, stop_t=0
)

# On extrait l'état au T=0 (Aujourd'hui)
V_today = V_history_poules[0]

def test_dp_poules(nom, gap_initial):
    gap_grid = int(gap_initial)
    # CORRECTION 4 : On ADDITIONNE l'offset pour trouver l'index
    idx_g1 = gap_grid + GAP_OFFSET
    
    wr_dp = V_today[idx_g1, idx_g2_safe, 0] 
    wr_th = exact_group_stage_wr(0, gap_initial)
    
    print(f"▶️ TEST : {nom}")
    print(f"   Retard sur Bob : {-gap_initial} pts")
    print(f"   WR Théorique   : {wr_th*100:05.2f}%")
    print(f"   WR Calculé DP  : {wr_dp*100:05.2f}%")
    
    if abs(wr_dp - wr_th) < 0.001:
        print("   ✅ TEST RÉUSSI")
    else:
        print("   ❌ ÉCHEC DU TEST")
    print("-" * 50)

# Gain max en 3 matchs (sans booster) = 270 pts
test_dp_poules("Mission Impossible (3 matchs)", gap_initial=-300)
test_dp_poules("Grand Chelem Requis (3 Outsiders)", gap_initial=-250)
test_dp_poules("Droit à l'erreur (2 Outsiders + 1 Nul)", gap_initial=-200)

🧪 DÉMARRAGE DU LABORATOIRE DP : PHASE DE POULES (NOTEBOOK 10)

🔍 PARTIE 1 : TEST DE compute_full_Q_table (1 Match)
▶️ TEST : Sauvetage Outsider x2
   Gap avec Bob : -150 pts
   [Base] Théorie : 00.00% | Calcul Q-Table : 00.00%
   [ x2 ] Théorie : 13.50% | Calcul Q-Table : 13.50%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Le Casse du Siècle (Nul x2 > Out x2)
   Gap avec Bob : -95 pts
   [Base] Théorie : 00.00% | Calcul Q-Table : 00.00%
   [ x2 ] Théorie : 20.00% | Calcul Q-Table : 20.00%
   ✅ TEST RÉUSSI
--------------------------------------------------

🌌 PARTIE 2 : TEST DE solve_dp_with_full_... (Multi-Matchs)
🧠 Rétro-propagation DP sur 3 matchs...
▶️ TEST : Mission Impossible (3 matchs)
   Retard sur Bob : 300 pts
   WR Théorique   : 00.00%
   WR Calculé DP  : 00.00%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Grand Chelem Requis (3 Outsiders)
   Retard sur Bob : 250 pts
   WR Théorique   : 00.25%
   WR Calculé DP 